# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Paper Finding 1: Refreshing stale content recovers search visibility
* **Label Origin:** Derived from post-refresh traffic changes observed over a 90-day window following an editorial update event.
* **Validation Assessment:** The claim is directionally supported by observed historical cohorts, but relies on a pre-post observational design. Without an explicit control group of un-updated decaying pages during the same calendar window, external macro shifts (e.g., search engine algorithm updates, seasonal demand) cannot be fully disentangled from the refresh action itself.

### Paper Finding 2: Low-CTR pages on Page 1 yield high-probability click lifts upon metadata updates
* **Label Origin:** CTR improvement observed on pages ranking in positions 1–10 following title/meta-description revisions.
* **Validation Assessment:** The label relies on short-term CTR delta. While the validation design is clean for ranking visibility, it assumes stable impression intent. If query intent shifts or SERP layout changes (e.g., addition of AI Overviews or featured snippets), CTR shifts may occur independently of metadata quality.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

# 1. Load Dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['clean_position'] = df['avg_position'].replace(0, np.nan)

# Target Proxy
med_ctr = df['ctr'].median()
df['target_is_decayed'] = ((df['trend_direction'] == 'down') & (df['ctr'] < med_ctr)).astype(int)

# Features
features = ['impressions_90d', 'clicks_90d', 'ctr', 'clean_position', 'content_age_days', 'word_count']
X = df[features].fillna(0)
y = df['target_is_decayed']
groups = df['client_id']

# --- SPLIT 1: Random Stratified Split (Week 5 Baseline) ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_random = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_random.fit(X_train_r, y_train_r)
probs_random = rf_random.predict_proba(X_test_r)[:, 1]
auc_random = roc_auc_score(y_test_r, probs_random)

# --- SPLIT 2: Honest Grouped Split (Grouped by client_id) ---
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups=groups))

X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_grouped.fit(X_train_g, y_train_g)
probs_grouped = rf_grouped.predict_proba(X_test_g)[:, 1]
auc_grouped = roc_auc_score(y_test_g, probs_grouped)

# Comparison Table
split_comparison = pd.DataFrame({
    'Validation Strategy': ['Random Stratified Split (Standard)', 'Client-Grouped Split (Honest Holdout)'],
    'ROC-AUC Score': [f"{auc_random:.4f}", f"{auc_grouped:.4f}"],
    'Performance Delta': ['Base', f"{auc_grouped - auc_random:+.4f}"]
})

print("--- VALIDATION SPLIT COMPARISON ---")
display(split_comparison)

--- VALIDATION SPLIT COMPARISON ---


,Validation Strategy,ROC-AUC Score,Performance Delta
0,Random Stratified Split (Standard),0.9392,Base
1,Client-Grouped Split (Honest Holdout),0.9142,-0.0249


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
# Feature Leakage Check across Final Feature Set
leakage_check = pd.DataFrame({
    'Feature Name': features,
    'Knowable at Decision Moment?': ['YES'] * len(features),
    'Contains Label Math?': ['NO'] * len(features),
    'Future Window Exposure?': ['NO'] * len(features),
    'Status': ['CLEAN / SAFE'] * len(features)
})

print("--- FINAL FEATURE LEAKAGE AUDIT ---")
display(leakage_check)

--- FINAL FEATURE LEAKAGE AUDIT ---


,Feature Name,Knowable at Decision Moment?,Contains Label Math?,Future Window Exposure?,Status
0,impressions_90d,YES,NO,NO,CLEAN / SAFE
1,clicks_90d,YES,NO,NO,CLEAN / SAFE
2,ctr,YES,NO,NO,CLEAN / SAFE
3,clean_position,YES,NO,NO,CLEAN / SAFE
4,content_age_days,YES,NO,NO,CLEAN / SAFE
5,word_count,YES,NO,NO,CLEAN / SAFE


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim Rewrite

* **Bold / Over-promising Statement (BEFORE):**
  > *"Our machine learning model accurately predicts which pages Google will downrank and guarantees traffic recovery upon refreshing content."*

* **Honest / Decision-Support Statement (AFTER):**
  > *"The Random Forest model identifies directional signals of organic performance decay on historical cohort data, serving as a decision-support tool to prioritize candidate pages for editorial review."*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.